# 成员6 正式版 DEA + K-Means Notebook（修订版）

本 Notebook 基于最新严格建模数据 `Final_Model_Dataset_Strict.csv`，完成 2020—2022 年省级医疗资源配置效率 DEA 正式测算，并对 2022 年结果进行 4 类 K-Means 聚类。

本修订版相较上一版，新增与修改内容如下：

1. 保留三年 DEA 正式建模结果；
2. 对 2022 年 Cluster 名称重新修订，确保 4 个 Cluster 对应 4 个不同名称；
3. 2022 年排名采用“先按 DEA_BCC、再按 DEA_CCR”排序；
4. 额外导出 2022 年前 10 / 后 10 省份结果；
5. 更适合直接写入 `成员6_算法模型与结果解读.docx`。

## 1. 导入所需库

本部分导入 DEA 求解、聚类分析、数据处理和绘图所需库。

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.optimize import linprog
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
mpl.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "SimSun"]
mpl.rcParams["axes.unicode_minus"] = False

## 2. 设置路径与参数

这里统一设置数据路径、输出路径、年份范围、随机种子和聚类类别数。

In [ ]:
BASE_DIR = Path(r"E:\2026_BigData_Project")
DATA_PATH = BASE_DIR / "data" / "processed" / "Final_Model_Dataset_Strict.csv"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS_TO_RUN = [2020, 2021, 2022]
LATEST_YEAR = 2022
RANDOM_SEED = 42
N_CLUSTERS = 4

INPUT_COLS = [
    "每千人口床位数(张/千人)",
    "卫生技术人员总数(人)",
    "人均医疗卫生财政支出(元)",
]

DESIRABLE_OUTPUT_COLS = ["预期寿命 (岁)"]
UNDESIRABLE_OUTPUT_COLS = ["孕产妇死亡率 (1/10 万)", "婴儿死亡率 (‰)"]
POSITIVE_OUTPUT_COLS = ["预期寿命 (岁)", "inv_孕产妇死亡率", "inv_婴儿死亡率"]

## 3. 读取数据并完成非期望产出正向化

正式模型中，孕产妇死亡率和婴儿死亡率属于“越低越好”的非期望产出，因此这里先做倒数正向化。

In [ ]:


def load_data(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"找不到数据文件：{csv_path}")
    df = pd.read_csv(csv_path)
    required_cols = ["地区", "年份"] + INPUT_COLS + DESIRABLE_OUTPUT_COLS + UNDESIRABLE_OUTPUT_COLS
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"缺少必要字段：{missing}")
    return df


def prepare_outputs(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-9
    out["inv_孕产妇死亡率"] = 1.0 / (out["孕产妇死亡率 (1/10 万)"].astype(float) + eps)
    out["inv_婴儿死亡率"] = 1.0 / (out["婴儿死亡率 (‰)"].astype(float) + eps)
    return out


df = load_data(DATA_PATH)
df = prepare_outputs(df)
print(df.shape)
df.head()


## 4. DEA 求解器

这里使用输入导向包络型 DEA。主模型为 BCC（变动规模报酬），并同时输出 CCR 与规模效率。

In [ ]:


def solve_dea_input_oriented(X: np.ndarray, Y: np.ndarray, rts: str = "bcc") -> np.ndarray:
    n, m = X.shape
    _, s = Y.shape
    eff_scores = np.full(n, np.nan)

    for o in range(n):
        c = np.zeros(n + 1)
        c[-1] = 1.0
        A_ub, b_ub = [], []

        for i in range(m):
            row = np.zeros(n + 1)
            row[:n] = X[:, i]
            row[-1] = -X[o, i]
            A_ub.append(row)
            b_ub.append(0.0)

        for r in range(s):
            row = np.zeros(n + 1)
            row[:n] = -Y[:, r]
            row[-1] = 0.0
            A_ub.append(row)
            b_ub.append(-Y[o, r])

        A_eq = None
        b_eq = None
        if rts.lower() == "bcc":
            A_eq = [np.r_[np.ones(n), 0.0]]
            b_eq = [1.0]

        bounds = [(0, None)] * n + [(0, 1)]
        res = linprog(
            c=c,
            A_ub=np.array(A_ub, dtype=float),
            b_ub=np.array(b_ub, dtype=float),
            A_eq=np.array(A_eq, dtype=float) if A_eq is not None else None,
            b_eq=np.array(b_eq, dtype=float) if b_eq is not None else None,
            bounds=bounds,
            method="highs",
        )
        if res.success:
            eff_scores[o] = res.x[-1]
    return eff_scores


def run_dea_one_year(df_year: pd.DataFrame) -> pd.DataFrame:
    result = df_year.copy().reset_index(drop=True)
    X = result[INPUT_COLS].astype(float).values
    Y = result[POSITIVE_OUTPUT_COLS].astype(float).values

    X_scale = np.where(X.mean(axis=0) == 0, 1.0, X.mean(axis=0))
    Y_scale = np.where(Y.mean(axis=0) == 0, 1.0, Y.mean(axis=0))
    X_scaled = X / X_scale
    Y_scaled = Y / Y_scale

    result["DEA_CCR"] = solve_dea_input_oriented(X_scaled, Y_scaled, rts="ccr")
    result["DEA_BCC"] = solve_dea_input_oriented(X_scaled, Y_scaled, rts="bcc")
    result["规模效率"] = (result["DEA_CCR"] / result["DEA_BCC"]).clip(upper=1.0)
    return result


def run_dea_all_years(df: pd.DataFrame, years: list[int]) -> pd.DataFrame:
    outputs = []
    for year in years:
        outputs.append(run_dea_one_year(df[df["年份"] == year].copy()))
    return pd.concat(outputs, ignore_index=True)


## 5. 运行 2020—2022 年 DEA 正式模型

In [ ]:


dea_all = run_dea_all_years(df, YEARS_TO_RUN)
print(dea_all.groupby("年份")[["DEA_CCR", "DEA_BCC", "规模效率"]].agg(["mean", "min", "max"]).round(6))


## 6. 2022 年 K-Means 聚类与标签修订

本次修订的关键点是：

- 不再让两个不同 Cluster 对应同一个名称；
- 通过“投入综合 z 值”和“效率综合 z 值”共同解释每一类；
- 最终形成 4 个可直接写入文档的类别名称。

In [ ]:


def build_cluster_profile(df_2022: pd.DataFrame):
    result = df_2022.copy().reset_index(drop=True)

    cluster_features = result[[
        "DEA_BCC", "规模效率",
        "每千人口床位数(张/千人)", "卫生技术人员总数(人)", "人均医疗卫生财政支出(元)",
        "预期寿命 (岁)", "inv_孕产妇死亡率", "inv_婴儿死亡率",
    ]].astype(float)

    X_std = StandardScaler().fit_transform(cluster_features)
    model = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=30)
    result["Cluster"] = model.fit_predict(X_std)

    profile = result.groupby("Cluster")[[
        "DEA_BCC", "规模效率",
        "每千人口床位数(张/千人)", "卫生技术人员总数(人)", "人均医疗卫生财政支出(元)",
        "预期寿命 (岁)", "inv_孕产妇死亡率", "inv_婴儿死亡率",
    ]].mean().reset_index()

    invest_features = ["每千人口床位数(张/千人)", "卫生技术人员总数(人)", "人均医疗卫生财政支出(元)"]
    eff_features = ["DEA_BCC", "规模效率", "预期寿命 (岁)", "inv_孕产妇死亡率", "inv_婴儿死亡率"]

    invest_z = pd.DataFrame(StandardScaler().fit_transform(result[invest_features]), columns=[f"z_{c}" for c in invest_features])
    eff_z = pd.DataFrame(StandardScaler().fit_transform(result[eff_features]), columns=[f"z_{c}" for c in eff_features])
    tmp_invest = pd.concat([result[["Cluster"]], invest_z], axis=1)
    tmp_eff = pd.concat([result[["Cluster"]], eff_z], axis=1)

    profile = profile.merge(tmp_invest.groupby("Cluster").mean().mean(axis=1).rename("投入综合均值_z"), on="Cluster")
    profile = profile.merge(tmp_eff.groupby("Cluster").mean().mean(axis=1).rename("效率综合均值_z"), on="Cluster")

    name_map = {}
    for _, row in profile.iterrows():
        cid = int(row["Cluster"])
        invest_z_mean = float(row["投入综合均值_z"])
        eff_z_mean = float(row["效率综合均值_z"])
        if invest_z_mean >= 0.8 and eff_z_mean >= 0.8:
            name = "高投入高效率核心优势区"
        elif invest_z_mean >= 0.0 and eff_z_mean <= -0.5:
            name = "高投入低效率洼地区"
        elif invest_z_mean <= -0.8 and eff_z_mean <= 0.0:
            name = "低投入低效率短板区"
        else:
            name = "中高投入稳健发展区"
        name_map[cid] = name

    if len(set(name_map.values())) < N_CLUSTERS:
        profile = profile.sort_values(["投入综合均值_z", "效率综合均值_z"], ascending=[False, False]).reset_index(drop=True)
        fallback_names = [
            "高投入高效率核心优势区",
            "中高投入稳健发展区",
            "高投入低效率洼地区",
            "低投入低效率短板区",
        ]
        name_map = {int(row["Cluster"]): fallback_names[i] for i, (_, row) in enumerate(profile.iterrows())}

    result["Cluster_Name"] = result["Cluster"].map(name_map)
    profile["Cluster_Name"] = profile["Cluster"].map(name_map)
    return result, profile.sort_values("Cluster").reset_index(drop=True)

latest = dea_all[dea_all["年份"] == LATEST_YEAR].copy().reset_index(drop=True)
final_2022, cluster_profile = build_cluster_profile(latest)
cluster_profile


## 7. 2022 年前 10 / 后 10 省份

这里采用“先按 DEA_BCC、再按 DEA_CCR、再按规模效率”的排序方式，以减少 BCC 大量并列时的可读性问题。

In [ ]:


def build_rankings(df_2022: pd.DataFrame):
    sort_cols = ["DEA_BCC", "DEA_CCR", "规模效率"]
    top10 = df_2022.sort_values(sort_cols, ascending=[False, False, False]).head(10).reset_index(drop=True)
    bottom10 = df_2022.sort_values(sort_cols, ascending=[True, True, True]).head(10).reset_index(drop=True)
    return top10, bottom10


top10, bottom10 = build_rankings(final_2022)
print("前10省份：")
display(top10[["地区", "DEA_BCC", "DEA_CCR", "规模效率", "Cluster_Name"]])
print("后10省份：")
display(bottom10[["地区", "DEA_BCC", "DEA_CCR", "规模效率", "Cluster_Name"]])


## 8. 生成三年均值表并导出全部结果

本部分会导出以下文件：

- `DEA_Scores_By_Year.csv`
- `DEA_Scores_Clusters_Final.csv`
- `DEA_Scores_Clusters_3YearMean.csv`
- `Cluster_Profile_2022.csv`
- `DEA_BCC_Top10_2022.csv`
- `DEA_BCC_Bottom10_2022.csv`

In [ ]:


def build_three_year_mean(df_all: pd.DataFrame) -> pd.DataFrame:
    keep_cols = [
        "DEA_CCR", "DEA_BCC", "规模效率",
        "每千人口床位数(张/千人)", "卫生技术人员总数(人)", "人均医疗卫生财政支出(元)",
        "预期寿命 (岁)", "孕产妇死亡率 (1/10 万)", "婴儿死亡率 (‰)",
        "inv_孕产妇死亡率", "inv_婴儿死亡率",
    ]
    out = df_all.groupby("地区")[keep_cols].mean().reset_index()
    out = out.sort_values(["DEA_BCC", "DEA_CCR"], ascending=[False, False]).reset_index(drop=True)
    return out

three_year_mean = build_three_year_mean(dea_all)

file_map = {
    "DEA_Scores_By_Year.csv": dea_all,
    "DEA_Scores_Clusters_Final.csv": final_2022,
    "DEA_Scores_Clusters_3YearMean.csv": three_year_mean,
    "Cluster_Profile_2022.csv": cluster_profile,
    "DEA_BCC_Top10_2022.csv": top10,
    "DEA_BCC_Bottom10_2022.csv": bottom10,
}

for fname, data in file_map.items():
    path = OUTPUT_DIR / fname
    data.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"已保存：{path}")


## 9. 最终结果摘要

下面给出适合直接用于组会和写作的摘要性输出。

In [ ]:

print("2022年各类数量：")
print(final_2022["Cluster_Name"].value_counts())

print("
聚类画像摘要：")
display(cluster_profile[["Cluster", "Cluster_Name", "投入综合均值_z", "效率综合均值_z", "DEA_BCC", "规模效率"]].round(6))


## 10. 阶段性解读

这版结果已经可以直接进入“正式分析与写作阶段”。

你接下来最适合做的事是：

1. 使用 `DEA_Scores_Clusters_Final.csv` 给成员5做地图和雷达图；
2. 使用 `DEA_BCC_Top10_2022.csv` 与 `DEA_BCC_Bottom10_2022.csv` 写“优势区/短板区”文字解读；
3. 使用 `Cluster_Profile_2022.csv` 写 4 类聚类画像说明；
4. 在 `成员6_算法模型与结果解读.docx` 中重点解释：
   - 为什么主模型选 BCC 输入导向；
   - 为什么规模效率普遍较高；
   - 为什么存在“高投入低效率洼地区”。